# Shared Trunk Model Playground
Same training framework as other playgrounds; this one uses the built-in shared trunk model.


In [1]:
from __future__ import annotations

from dataclasses import replace
from datetime import datetime, timezone
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "train_headless.py").exists() and (REPO_ROOT.parent / "train_headless.py").exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from brains.config import load_runtime_spec
from brains.jax_trainer import apply_runtime_spec
from brains.models import (
    NotebookModel,
    apply_notebook_model,
    get_model_definition,
    register_notebook_model,
)
from brains.models.notebook_framework import train_and_save_with_progress


In [ ]:
MODEL_TYPE = "notebook_shared_trunk_v1"
LOG_ID = datetime.now(timezone.utc).strftime("shared_trunk_%Y%m%dT%H%M%SZ")
TRAIN_GENERATIONS = 100
SEED = 7

# smoke.yaml: short episodes / small population — playground sanity, not real training.
# Switch to configs/default.yaml for real ES runs (or use train_headless.py).
base_def = get_model_definition("shared_trunk_es")
base_spec = load_runtime_spec(REPO_ROOT / "configs/default.yaml")
model = NotebookModel(
    model_type=MODEL_TYPE,
    architecture=base_def.architecture,
    description="Local shared-trunk baseline registered from notebook.",
    input_size=base_def.input_size,
    output_size=base_def.output_size,
    parameter_count=base_def.parameter_count,
    policy_entrypoint=None,
    control_mode="motor_targets",
)
register_notebook_model(model, spec=base_spec, persist=True)

spec = apply_notebook_model(base_spec, model)
spec = replace(spec, name="shared-trunk-default")
apply_runtime_spec(spec)

{
    "model": spec.model.type,
    "architecture": spec.model.architecture,
    "control_mode": spec.control.mode,
    "train_generations": TRAIN_GENERATIONS,
}


{'model': 'notebook_shared_trunk_v1',
 'architecture': 'shared_trunk_motor_lanes',
 'control_mode': 'motor_targets',
 'train_generations': 2}

In [3]:
result = train_and_save_with_progress(
    spec=spec,
    repo_root=REPO_ROOT,
    model_type=MODEL_TYPE,
    log_id=LOG_ID,
    seed=SEED,
    generations=TRAIN_GENERATIONS,
    print_step_progress=False,
)
result



W0501 11:26:22.422723 2675852 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


generation 1/2 start


generation 1/2 done | mean_reward=-0.6227 | best_reward=-0.1660


generation 2/2 start


generation 2/2 done | mean_reward=1.7376 | best_reward=1.1314


{'model_id': 'notebook_shared_trunk_v1_shared_trunk_20260501T152622Z',
 'log_id': 'shared_trunk_20260501T152622Z',
 'latest': '/Users/monicagraham/Desktop/GitHub/multi-brain-quadruped-sim/checkpoints/notebook_shared_trunk_v1_shared_trunk_20260501T152622Z/latest.npz',
 'generation': 2,
 'mean_reward': 1.7375733852386475,
 'best_reward': 1.1313796043395996,
 'elapsed_s': 122.85202641600335,
 'visible_artifacts': ['notebook_shared_trunk_v1_shared_trunk_20260501T152622Z',
  'notebook_cnn_patch_v1_cnn_patch_20260501T151038Z',
  'notebook_simple_snn_v1_simple_snn_20260501T151036Z',
  'notebook_command_brain_v1_command_brain_20260501T151039Z',
  'notebook_shared_trunk_v1_shared_trunk_20260501T130232Z']}